> **Prerequisite:** Run earlier notebooks in numeric order in the **same kernel session**.

# 02 — Panel plotting functions

Style constants and drawing routines for panels a–e.

In [1]:
"""Panel plotting for macro-scale gradients figure (HMM states 3-6)."""

from __future__ import annotations


import re

import warnings

from pathlib import Path


import matplotlib as mpl

import matplotlib.patches as mpatches

import matplotlib.pyplot as plt

from matplotlib.transforms import blended_transform_factory

import numpy as np

import pandas as pd

from scipy import stats as scipy_stats

from statsmodels.api import OLS, add_constant


warnings.filterwarnings("ignore")


# ── Publication parameter block ─────────────────────────────────────────────
WIDTH_MM, HEIGHT_MM, DPI = 270, 168, 600
FS, FS_TICK, FS_PANEL, FS_SMALL = 10, 9, 13, 8.5

WONG = ["#E69F00", "#56B4E9", "#009E73", "#CC79A7", "#D55E00", "#F0E442", "#0072B2", "#000000"]
SN_RED = WONG[4]
SN_RED_DARK = "#A84300"
SN_RED_MID = "#D55E00"
SN_RED_LIGHT = "#F4A66D"
NET_GRAY = "#737373"
GRAY_LIGHT = "#E8E8E8"
TEXT = "#1A1A1A"

NET_COLORS = {
"Vis": "#4D4D4D",
"SomMot": "#999999",
"DorsAttn": "#56B4E9",
"SalVentAttn": SN_RED,
"Limbic": "#BFBFBF",
"Cont": "#009E73",
"Default": "#CC79A7",
}
BLUE = NET_COLORS["DorsAttn"]  # match panel a DorsAttn blue
BLUE_LIGHT = "#C8E8F8"
# panel a SN highlight - Paul Tol wine, colorblind-safe (not blue / green / pink / orange)
SN_PANEL_A = "#882255"
PANEL_A_XLIM = (-2.6, 2.5)
PANEL_A_YLIM = (-2.0, 2.0)
PANEL_A_XTICKS = np.arange(-2.6, 2.6, 1.7)
PANEL_A_YTICKS = np.arange(-2, 2.1, 1.0)
PANEL_A_BG_S = 18.0
PANEL_A_SN_S = 21.0  # SN highlight - unchanged
PANEL_A_ANNOT_MIN_SEP_PT = 34.0
PANEL_A_NET_LABELS: tuple[tuple[str, str], ...] = (
("SalVentAttn", "SN"),
)
PANEL_A_SN_LABEL_XY = (-0.05, 0.04)
PANEL_A_SCHEMATIC_HINTS: dict[str, tuple[float, float]] = {
"SalVentAttn": PANEL_A_SN_LABEL_XY,
}
PANEL_A_X_SHIFT_PT = 10
PANEL_A_TICK_LEN_PT = 4
AXIS_SHIFT_PT = 10
AXIS_TICK_LEN_PT = 4
PANEL_B_XLIM = (-1.9, 3.2)
PANEL_B_YLIM = (-3.0, 3.0)
PANEL_B_XTICKS = np.arange(-1.9, 3.2 + 1e-9, 1.7)
PANEL_B_YTICKS = np.arange(-3, 3.1, 2.0)
PANEL_B_FOREST_XTICKS = [-0.8, 0.0, 0.8]
PANEL_B_FOREST_W, PANEL_B_FOREST_H = 0.24, 0.17
PANEL_B_FOREST_ALIGN_X = 3.2
PANEL_B_FOREST_ALIGN_Y = 3.0
PANEL_C_XLIM = (-1.8, 2.4)
PANEL_C_YLIM = (-2.7, 2.5)
PANEL_C_XTICKS = np.arange(-1.8, 2.4 + 1e-9, 1.4)
PANEL_C_YTICKS = np.arange(-2.7, 2.5 + 1e-9, 1.3)
PANEL_C_MID = "#44AA99"
PANEL_C_DARK = "#1A6B62"
PANEL_D_W_BOUND = 0.070
PANEL_D_WLIM = (-2.0, 2.2)
PANEL_D_YLIM = (-2.0, 2.0)
PANEL_D_YTICKS = np.arange(-2.0, 2.0 + 1e-9, 1.0)
PANEL_D_BRAIN_W, PANEL_D_BRAIN_H = 0.17, 0.24
PANEL_D_BRAIN_LEFT = (0.03, 0.76, PANEL_D_BRAIN_W, PANEL_D_BRAIN_H)
PANEL_D_BRAIN_RIGHT = (0.795, 0.007, PANEL_D_BRAIN_W, PANEL_D_BRAIN_H)
PANEL_D_LABEL_GAP = 0.040
PANEL_D_LABEL_TITLE_SUB_GAP = 0.064
PANEL_D_SUBTITLE_INTERNAL_GAP = 0.052
PANEL_D_TEXT_SHIFT_UP = 0.050
PANEL_D_RIGHT_SUBTITLE_SHIFT_DOWN = 0.075
PANEL_D_REGION_TEXT_SHIFT_DOWN = 0.038
PANEL_D_RIGHT_LABEL_SHIFT_UP = 0.035
PANEL_D_RIGHT_TITLE_SHIFT_DOWN = 0.012
PANEL_LABEL_Y_OFFSET = 0.008
PANEL_LABEL_Y_OFFSET_ABC = -0.003

PANEL_E_XLIM = (0.0, 15.8)
PANEL_E_YLIM = (0.0, 10.0)
PANEL_E_BOX_SCALE = 1.5
PANEL_E_GAP_SCALE = 2.0
PANEL_E_BOX_W_SCALE = 0.714
PANEL_E_BOX_H_SCALE = 1.428
PANEL_E_GAP_SPACING_SCALE = 1.2
PANEL_E_BOX_W_NARROW = 0.9
PANEL_E_BOX_W_MULT = 1.452
PANEL_E_GAP_H_MULT = 1.452
PANEL_E_ARR_PAD = 0.20328
PANEL_E_IMM_FS = 11
PANEL_E_LAYOUT = {
"d_left_boxes_dx": 0.45,
"d_first_box_dx": 0.28,
"d_content_dx_bias": -0.42,
"d_box_alpha": 0.52,
"d_beta_y_offset": 0.34,
"d_w_box_dy": -1.38,
"d_int_label_dx": 0.08,
"d_int_arrow_gap": 0.18,
"d_path_a_stat_dy": 0.52,
"d_path_a_lbl_pad": 0.14,
"d_note_line_dy": 0.76,
"d_note_y_up": 0.70,
}

OUT_BASENAME = "macro_gradients_topographic_fidelity"
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / 'data').is_dir():
    GRADIENT_DISPERSION_MEDIATE_MODERATE_ROOT = NOTEBOOK_DIR
else:
    GRADIENT_DISPERSION_MEDIATE_MODERATE_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = GRADIENT_DISPERSION_MEDIATE_MODERATE_ROOT / 'data'
FIGURES_DIR = GRADIENT_DISPERSION_MEDIATE_MODERATE_ROOT / 'figures'
OUT_DIR = FIGURES_DIR

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": FS,
        "axes.linewidth": 0.75,
        "axes.labelsize": FS,
        "xtick.labelsize": FS_TICK,
        "ytick.labelsize": FS_TICK,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "figure.facecolor": "none",
        "axes.facecolor": "none",
        "savefig.facecolor": "none",
        "savefig.edgecolor": "none",
        "savefig.transparent": True,
        "savefig.dpi": DPI,
    }
)

_TITLE_CASE_SMALL = frozenset({
    "a", "an", "the", "and", "or", "but", "nor", "to", "of", "in", "on", "at", "by",
    "for", "with", "as", "from", "into", "via", "vs", "per", "if", "is", "are",
})


def _cap_token(word: str, *, force: bool) -> str:
    if not word or word.startswith("\\") or "{" in word or "$" in word:
        return word
    if word.isupper() and len(word) <= 4:
        return word
    lower = word.lower()
    if not force and lower in _TITLE_CASE_SMALL:
        return lower
    if "-" in word:
        parts = word.split("-")
        return "-".join(_cap_token(p, force=force and i == 0) for i, p in enumerate(parts))
    return lower.capitalize()


def _title_case_prose(text: str) -> str:
    def _proc_segment(segment: str) -> str:
        lines_out: list[str] = []
        for line in segment.split("\n"):
            tokens = re.findall(r"\S+", line)
            if not tokens:
                lines_out.append(line)
                continue
            rebuilt: list[str] = []
            for i, tok in enumerate(tokens):
                m = re.match(r"^([(\[]*)(.*?)([\)\],;:]*)$", tok)
                lead, core, trail = m.groups()
                force = i == 0 or core.endswith(":")
                rebuilt.append(f"{lead}{_cap_token(core, force=force)}{trail}")
            lines_out.append(" ".join(rebuilt))
        return "\n".join(lines_out)

    parts = re.split(r"(\$[^$]+\$)", text)
    return "".join(_proc_segment(p) if not p.startswith("$") else p for p in parts)


def _place_panel_labels(
fig: plt.Figure,
axes: list[plt.Axes],
letters: str,
ref_ax: plt.Axes,
*,
y_offset: float | None = None,
) -> None:
    """Left-align panel labels with the left edge of the reference panel Y-axis title."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    x_fig = fig.transFigure.inverted().transform(
    (ref_ax.yaxis.label.get_window_extent(renderer).x0, 0)
    )[0]
    y_off = PANEL_LABEL_Y_OFFSET if y_offset is None else y_offset
    for ax, letter in zip(axes, letters, strict=True):
        y_fig = ax.get_position().y1 + y_off
        fig.text(
        x_fig, y_fig, letter,
        transform=fig.transFigure, ha="left", va="bottom",
        fontsize=FS_PANEL, fontweight="bold",
        )


def _sig_stars(p: float) -> str:
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def _compute_panel_e_layout(content_dx: float | None = None) -> dict:
    """Shared E-panel mediation-diagram geometry."""
    lay = PANEL_E_LAYOUT
    s, g = PANEL_E_BOX_SCALE, PANEL_E_GAP_SCALE
    w_scale = PANEL_E_BOX_W_SCALE * PANEL_E_BOX_W_NARROW
    h_scale = PANEL_E_BOX_H_SCALE
    box_w_scale = 1.1 * 0.9 * PANEL_E_BOX_W_MULT
    bw_x = 2.4 * s * w_scale * box_w_scale
    bh = 1.1 * s * h_scale
    bw_y = 2.5 * s * w_scale * box_w_scale
    bh_y = 1.1 * s * h_scale
    bw_w = 2.6 * s * w_scale * box_w_scale
    bh_w = 1.0 * s * h_scale
    gap_h = 0.8 * g * PANEL_E_GAP_SPACING_SCALE * PANEL_E_GAP_H_MULT
    gap_v = 1.0 * g * PANEL_E_GAP_SPACING_SCALE
    y_top = 6.0
    x0_x = 0.6 - lay["d_left_boxes_dx"] - lay["d_first_box_dx"]
    x0_m = x0_x + bw_x + gap_h
    x0_y = x0_m + bw_x + gap_h
    y0_w = y_top - gap_v - bh_w + lay["d_w_box_dy"]
    x0_w = (x0_m + bw_x / 2 + x0_y + bw_y / 2 - bw_w) / 2
    content_left = min(x0_x, x0_w)
    content_right = x0_y + bw_y
    content_bottom = y0_w - 0.35
    imm_y = y_top + bh + 0.65
    content_top = imm_y + 0.55
    content_cx = (content_left + content_right) / 2
    content_cy = (content_bottom + content_top) / 2
    dx = lay["d_content_dx_bias"] if content_dx is None else content_dx
    ox = (PANEL_E_XLIM[0] + PANEL_E_XLIM[1]) / 2 - content_cx + dx
    oy = (PANEL_E_YLIM[0] + PANEL_E_YLIM[1]) / 2 - content_cy
    return {
    "x0_x": x0_x, "x0_m": x0_m, "x0_y": x0_y, "x0_w": x0_w,
    "y_top": y_top, "y0_w": y0_w, "bw_x": bw_x, "bh": bh,
    "bw_y": bw_y, "bh_y": bh_y, "bw_w": bw_w, "bh_w": bh_w,
    "content_left": content_left,
    "ox": ox, "oy": oy,
    "path_b_mid_x": (x0_m + bw_x + x0_y) / 2,
    "content_cx": content_cx, "content_cy": content_cy,
    }

def _ax_data_y_from_fig_y(ax: plt.Axes, y_fig: float) -> float:
    pos = ax.get_position()
    y_frac = (y_fig - pos.y0) / pos.height
    return float(ax.transData.inverted().transform(ax.transAxes.transform((0.5, y_frac)))[1])

def _panel_a_limits_from_data(
    grad_df: pd.DataFrame,
) -> tuple[tuple[float, float], tuple[float, float], np.ndarray, np.ndarray]:
    """Axis limits and ticks from empirical G1/G2 (BrainSpace scale, not schematic)."""
    g1 = grad_df["g1"].to_numpy(dtype=float)
    g2 = grad_df["g2"].to_numpy(dtype=float)

    def padded(v: np.ndarray) -> tuple[float, float]:
        lo, hi = np.nanpercentile(v, [1, 99])
        span = float(hi - lo)
        pad = max(span * 0.14, 0.008)
        return float(lo - pad), float(hi + pad)

    xlim = padded(g1)
    ylim = padded(g2)
    xticks = np.linspace(xlim[0], xlim[1], 4)
    yticks = np.linspace(ylim[0], ylim[1], 5)
    return xlim, ylim, xticks, yticks


def _panel_a_annot_hints(
grad_df: pd.DataFrame,
xmin: float,
xmax: float,
ymin: float,
ymax: float,
) -> dict[str, tuple[float, float]]:
    """Label anchor positions as offsets from each network median (data coordinates)."""
    span_x = xmax - xmin
    span_y = ymax - ymin
    offsets = {
    "Vis": (-0.45 * span_x, 0.24 * span_y),
    "Default": (0.45 * span_x, 0.24 * span_y),
    "SomMot": (-0.40 * span_x, -0.36 * span_y),
    "DorsAttn": (0.06 * span_x, -0.40 * span_y),
    "Cont": (0.38 * span_x, 0.06 * span_y),
    "SalVentAttn": (0.16 * span_x, 0.38 * span_y),
    "Limbic": (-0.14 * span_x, -0.18 * span_y),
    }
    hints: dict[str, tuple[float, float]] = {}
    for net, (ox, oy) in offsets.items():
        sub = grad_df[grad_df["network"] == net]
        if sub.empty:
            continue
        cx, cy = float(sub["g1"].median()), float(sub["g2"].median())
        hints[net] = (cx + ox, cy + oy)
    return hints


def _load_margulies_schematic() -> pd.DataFrame:
    """Schematic G1/G2 layout (fallback when empirical embedding CSV is absent)."""
    from nilearn import datasets

    atlas = datasets.fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=7, resolution_mm=2)
    labels = np.loadtxt(atlas["labels"], dtype=str)[1:]
    net_map = {"Vis": "Vis", "SomMot": "SomMot", "DorsAttn": "DorsAttn",
               "SalVentAttn": "SalVentAttn", "Cont": "Cont", "Default": "Default"}
    g1_centers = {"Vis": -2.2, "SomMot": -1.4, "DorsAttn": -0.2, "SalVentAttn": 0.35,
                  "Cont": 1.0, "Default": 2.0}
    g2_centers = {"Vis": 1.6, "SomMot": -0.4, "DorsAttn": -1.2, "SalVentAttn": 0.15,
                  "Cont": 0.5, "Default": 1.3}
    rng = np.random.RandomState(7)
    rows = []
    for i, lab in enumerate(labels):
        net = next((k for k in net_map if k in lab), "Cont")
        jitter = (i % 17) / 17.0 - 0.5
        g1 = g1_centers[net] + 0.22 * jitter + rng.normal(0, 0.06)
        g2 = g2_centers[net] + 0.18 * jitter + rng.normal(0, 0.05)
        rows.append({"parcel": i, "network": net, "g1": g1, "g2": g2, "is_sn": net == "SalVentAttn"})
    return pd.DataFrame(rows)


def _load_panel_a_embedding() -> tuple[pd.DataFrame, str, bool]:
    """Load bundled panel-a G1/G2 parcel embedding; schematic fallback if missing."""
    panel_csv = DATA_DIR / "panel_a_g12_parcels.csv"
    if panel_csv.is_file():
        df = pd.read_csv(panel_csv)
        return df, f"empirical ({panel_csv.name}, n={len(df)} parcels)", True
    print("panel a: bundled CSV missing; using schematic fallback.")
    return _load_margulies_schematic(), "schematic fallback", False

def _net_median(grad_df: pd.DataFrame, net: str) -> tuple[float, float]:
    sub = grad_df[grad_df["network"] == net]
    return float(sub["g1"].median()), float(sub["g2"].median())


def _axis_x_shift(
ax: plt.Axes, x_min: float, y_min: float, shift_pt: float | None = None,
) -> float:
    shift_pt = AXIS_SHIFT_PT if shift_pt is None else shift_pt
    ax.figure.canvas.draw()
    p0 = ax.transData.transform((x_min, y_min))
    scale = ax.figure.dpi / 72.0
    gx, _ = ax.transData.inverted().transform((p0[0] + shift_pt * scale, p0[1]))
    return float(gx - x_min)


def _axis_tick_len_data(ax: plt.Axes, xmin: float, ymin: float) -> float:
    p0 = ax.transData.transform((xmin, ymin))
    p1 = (p0[0], p0[1] - AXIS_TICK_LEN_PT * ax.figure.dpi / 72.0)
    _, y1 = ax.transData.inverted().transform(p1)
    return float(ymin - y1)


def _draw_disconnected_axes(
    ax: plt.Axes, xmin: float, xmax: float, ymin: float, ymax: float, x_shift: float,
) -> None:
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(
        axis="both", top=False, right=False, bottom=True, left=True,
        direction="out", length=0, width=0.8, colors=TEXT, pad=8,
    )
    kw = dict(color=TEXT, lw=0.9, clip_on=False, zorder=6, solid_capstyle="butt")
    ax.plot([xmin, xmin], [ymin, ymax], **kw)
    ax.plot([xmin + x_shift, xmax + x_shift], [ymin, ymin], **kw)


def _outward_tick_marks(
    ax: plt.Axes, xmin: float, ymin: float, x_shift: float,
) -> None:
    tl = _axis_tick_len_data(ax, xmin, ymin)
    tk = dict(color=TEXT, lw=0.8, clip_on=False, zorder=8, solid_capstyle="butt")
    for x in ax.get_xticks():
        ax.plot([x, x], [ymin, ymin - tl], **tk)
    for y in ax.get_yticks():
        ax.plot([xmin - tl, xmin], [y, y], **tk)


def _panel_a_x_shift(
ax: plt.Axes, xmin: float, ymin: float, shift_pt: float | None = None,
) -> float:
    return _axis_x_shift(ax, xmin, ymin, shift_pt or PANEL_A_X_SHIFT_PT)


def _panel_a_format_x_tick(t: float) -> str:
    return f"{t:.2f}"


def _panel_a_format_y_tick(t: float, *, empirical: bool = False) -> str:
    if empirical:
        if abs(t) < 0.1:
            return f"{t:.2f}"
        return f"{t:.1f}" if abs(t - round(t)) > 0.05 else f"{t:.0f}"
    return f"{t:.0f}" if abs(t - round(t)) < 1e-6 else f"{t:.1f}"


def _panel_a_data_step_for_pt(
ax: plt.Axes, origin: tuple[float, float], ux: float, uy: float, len_pt: float,
) -> float:
    """Data-space step along (ux, uy) whose display length equals len_pt."""
    ox, oy = origin
    p0 = ax.transData.transform((ox, oy))
    p1 = ax.transData.transform((ox + ux, oy + uy))
    per_unit = float(np.linalg.norm(p1 - p0))
    return len_pt / per_unit if per_unit > 1e-9 else 0.0


def _panel_a_ref_arrow_len_pt(
ax: plt.Axes, anchor: tuple[float, float], text_pos: tuple[float, float],
) -> float:
    p0 = ax.transData.transform(anchor)
    p1 = ax.transData.transform(text_pos)
    return float(np.linalg.norm(p1 - p0))


def _panel_a_tail_position(
    ax: plt.Axes,
    cx: float,
    cy: float,
    hint: tuple[float, float],
    len_pt: float,
    x_shift: float,
) -> tuple[float, float]:
    hx, hy = hint
    dx, dy = hx - cx, hy - cy
    dist = float(np.hypot(dx, dy))
    if dist < 1e-9:
        return cx + x_shift, cy
    ux, uy = dx / dist, dy / dist
    cx_s = cx + x_shift
    step = _panel_a_data_step_for_pt(ax, (cx_s, cy), ux, uy, len_pt)
    return cx_s + ux * step, cy + uy * step


def _panel_a_spread_hints(
    hint_map: dict[str, tuple[float, float]],
    centroids: dict[str, tuple[float, float]],
    ax: plt.Axes,
    x_shift: float,
    arrow_len_pt: float,
    xmin: float,
    xmax: float,
    ymin: float,
    ymax: float,
    *,
    min_sep_pt: float = PANEL_A_ANNOT_MIN_SEP_PT,
    max_iter: int = 30,
) -> dict[str, tuple[float, float]]:
    """Nudge label anchor hints until arrow tails are separated in display space."""
    hints = {k: (float(v[0]), float(v[1])) for k, v in hint_map.items()}
    nets = [n for n in hints if n in centroids]
    if len(nets) < 2:
        return hints
    span = max(xmax - xmin, ymax - ymin, 1e-6)
    nudge = span * 0.035
    ax.figure.canvas.draw()
    for _ in range(max_iter):
        moved = False
        tails = {
            n: _panel_a_tail_position(ax, *centroids[n], hints[n], arrow_len_pt, x_shift)
            for n in nets
        }
        disp = {n: np.asarray(ax.transData.transform(tails[n])) for n in nets}
        for i, n1 in enumerate(nets):
            for n2 in nets[i + 1:]:
                delta = disp[n1] - disp[n2]
                sep = float(np.linalg.norm(delta))
                if sep >= min_sep_pt:
                    continue
                if sep < 1e-6:
                    dx, dy = 1.0, 0.0
                else:
                    dx, dy = delta[0] / sep, delta[1] / sep
                # push in data space along hint-centroid directions
                for n, sign in ((n1, 1.0), (n2, -1.0)):
                    cx, cy = centroids[n]
                    hx, hy = hints[n]
                    vx, vy = hx - cx, hy - cy
                    vlen = float(np.hypot(vx, vy))
                    if vlen < 1e-9:
                        vx, vy = dx * sign, dy * sign
                    else:
                        vx, vy = vx / vlen, vy / vlen
                    hints[n] = (hx + vx * nudge, hy + vy * nudge)
                    moved = True
        if not moved:
            break
    return hints


def _panel_a_draw_network_labels(
    ax: plt.Axes,
    grad_df: pd.DataFrame,
    *,
    empirical: bool,
    xmin: float,
    xmax: float,
    ymin: float,
    ymax: float,
    x_shift: float,
) -> None:
    centroids = {
        net: _net_median(grad_df, net)
        for net, _ in PANEL_A_NET_LABELS
        if (grad_df["network"] == net).any()
    }
    if not centroids:
        return

    if empirical:
        hint_map = _panel_a_annot_hints(grad_df, xmin, xmax, ymin, ymax)
    else:
        hint_map = {
            net: PANEL_A_SCHEMATIC_HINTS[net]
            for net in centroids
            if net in PANEL_A_SCHEMATIC_HINTS
        }

    if "SalVentAttn" in centroids:
        hint_map["SalVentAttn"] = PANEL_A_SN_LABEL_XY

    ref_net = "SalVentAttn" if "SalVentAttn" in centroids else next(iter(centroids))
    rcx, rcy = centroids[ref_net]
    ref_hint = hint_map.get(ref_net, (xmin, ymin))
    arrow_len_pt = _panel_a_ref_arrow_len_pt(
        ax, (rcx + x_shift, rcy), (ref_hint[0] + x_shift, ref_hint[1]),
    ) / 2.0
    hint_map = _panel_a_spread_hints(
        hint_map, centroids, ax, x_shift, arrow_len_pt, xmin, xmax, ymin, ymax,
    )

    for net, label in PANEL_A_NET_LABELS:
        if net not in centroids or net not in hint_map:
            continue
        cx, cy = centroids[net]
        _panel_a_draw_arrow_label(
            ax, (cx, cy), hint_map[net], label, arrow_len_pt, x_shift,
        )


def _panel_a_draw_arrow_label(
    ax: plt.Axes,
    xy: tuple[float, float],
    dir_hint: tuple[float, float],
    label: str,
    len_pt: float,
    x_shift: float,
) -> None:
    """Label at arrow tail; vertical edge aligned to tail (above→bottom, below→top)."""
    cx, cy = xy
    hx, hy = dir_hint
    dx, dy = hx - cx, hy - cy
    dist = float(np.hypot(dx, dy))
    if dist < 1e-9:
        return
    ux, uy = dx / dist, dy / dist
    cx_s = cx + x_shift
    arrow_step = _panel_a_data_step_for_pt(ax, (cx_s, cy), ux, uy, len_pt)
    tail_x = cx_s + ux * arrow_step
    tail_y = cy + uy * arrow_step
    ax.add_patch(
        mpatches.FancyArrowPatch(
            (tail_x, tail_y), (cx_s, cy),
            arrowstyle="-|>",
            color=TEXT,
            lw=0.7,
            shrinkA=0,
            shrinkB=0,
            mutation_scale=7,
            zorder=20,
            clip_on=False,
        )
    )
    va = "bottom" if hy >= cy else "top"
    ax.text(
        tail_x, tail_y, label, fontsize=FS_SMALL, ha="center", va=va,
        color=TEXT, zorder=21, clip_on=False,
    )


def panel_a(ax: plt.Axes, grad_df: pd.DataFrame, *, empirical: bool = False) -> None:
    if empirical:
        (xmin, xmax), (ymin, ymax), x_ticks, y_ticks = _panel_a_limits_from_data(grad_df)
    else:
        xmin, xmax = PANEL_A_XLIM
        ymin, ymax = PANEL_A_YLIM
        x_ticks = PANEL_A_XTICKS
        y_ticks = PANEL_A_YTICKS

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.figure.canvas.draw()
    x_shift = _panel_a_x_shift(ax, xmin, ymin)

    bg = grad_df[~grad_df["is_sn"]]
    sn = grad_df[grad_df["is_sn"]]
    for net in ["Vis", "SomMot", "DorsAttn", "Limbic", "Cont", "Default"]:
        sub = bg[bg["network"] == net]
        if sub.empty:
            continue
        ax.scatter(
            sub["g1"] + x_shift, sub["g2"], s=PANEL_A_BG_S, c=NET_COLORS[net],
            alpha=0.28, linewidths=0, rasterized=True,
        )
    if not sn.empty:
        ax.scatter(
            sn["g1"] + x_shift, sn["g2"], s=PANEL_A_SN_S, c=SN_PANEL_A, alpha=0.92,
            edgecolors="white", linewidths=0.15, zorder=3, rasterized=True,
        )

    ax.set_xlabel(_title_case_prose("Functional Gradient 1 (unimodal $\\rightarrow$ transmodal)"))
    ax.set_ylabel(_title_case_prose("Functional Gradient 2"))
    ax.set_xticks(x_ticks + x_shift)
    ax.set_xticklabels([_panel_a_format_x_tick(t) for t in x_ticks])
    ax.set_yticks(y_ticks)
    ax.set_yticklabels([_panel_a_format_y_tick(t, empirical=empirical) for t in y_ticks])
    ax.set_xlim(xmin, xmax + x_shift)
    ax.figure.canvas.draw()

    _panel_a_draw_network_labels(
        ax, grad_df,
        empirical=empirical,
        xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax,
        x_shift=x_shift,
    )

    _draw_disconnected_axes(ax, xmin, xmax, ymin, ymax, x_shift)
    _outward_tick_marks(ax, xmin, ymin, x_shift=0)


def _format_axis_tick(t: float) -> str:
    return f"{t:.0f}" if abs(t - round(t)) < 0.05 else f"{t:.1f}"


def _panel_b_forest_bbox_aligned(
ax: plt.Axes, x_shift: float, x_tick: float, y_tick: float,
) -> tuple[float, float, float, float]:
    """Forest inset: right edge at x_tick; top edge at y_tick (axes-fraction placement)."""
    ax.figure.canvas.draw()
    w, h = PANEL_B_FOREST_W, PANEL_B_FOREST_H
    x_frac = ax.transAxes.inverted().transform(
    ax.transData.transform((x_tick + x_shift, PANEL_B_YLIM[0]))
    )[0]
    left = float(x_frac - w)
    left = max(0.02, min(left, 1.0 - w - 0.002))
    y_frac_top = ax.transAxes.inverted().transform(
    ax.transData.transform((PANEL_B_XLIM[0], y_tick))
    )[1]
    bottom = float(y_frac_top - h)
    bottom = max(0.02, min(bottom, 1.0 - h - 0.002))
    return left, bottom, w, h


def _panel_b_forest_inset(ax: plt.Axes, model, x_shift: float) -> None:
    """Compact horizontal forest plot for standardized regression coefficients."""
    inset = ax.inset_axes(
        _panel_b_forest_bbox_aligned(ax, x_shift, PANEL_B_FOREST_ALIGN_X, PANEL_B_FOREST_ALIGN_Y)
    )
    ci = np.asarray(model.conf_int())
    betas = [float(model.params.iloc[1]), float(model.params.iloc[2])]
    cis = [(float(ci[1, 0]), float(ci[1, 1])), (float(ci[2, 0]), float(ci[2, 1]))]
    labels = [_title_case_prose("Transmodal"), _title_case_prose("Unimodal")]
    cols = [SN_RED_MID, NET_GRAY]
    ypos = np.array([1.0, 0.0])
    x_lo, x_hi = PANEL_B_FOREST_XTICKS[0], PANEL_B_FOREST_XTICKS[-1]
    for yi, b, (lo, hi), col, lab in zip(ypos, betas, cis, cols, labels, strict=False):
        inset.errorbar(
            b, yi,
            xerr=[[b - lo], [hi - b]],
            fmt="D", ms=3.75, mew=0.6,
            color=col, markerfacecolor=col, markeredgecolor=col, ecolor=col,
            capsize=0, lw=0.95, zorder=3,
        )
        inset.text(
            -0.06, yi, lab, transform=inset.get_yaxis_transform(),
            ha="right", va="center", fontsize=FS_SMALL, color=TEXT,
        )
    inset.axvline(0, color=TEXT, ls=(0, (4.0, 3.0)), lw=0.65, zorder=1)
    inset.set_xlim(x_lo, x_hi)
    inset.set_ylim(-0.55, 1.55)
    inset.set_yticks([])
    inset.set_xlabel("$\\beta$ (95% CI)", fontsize=FS_SMALL, labelpad=2)
    inset.set_xticks(PANEL_B_FOREST_XTICKS)
    inset.set_xticklabels([f"{v:.1f}" for v in PANEL_B_FOREST_XTICKS])
    inset.tick_params(axis="x", labelsize=FS_TICK, pad=2, length=2.5, width=0.65, colors=TEXT)
    inset.spines["top"].set_visible(False)
    inset.spines["right"].set_visible(False)
    inset.spines["left"].set_visible(False)
    inset.tick_params(axis="y", length=0, labelleft=False)


def panel_b(ax: plt.Axes, reg: dict) -> None:
    sub = reg["sub_z"]
    model = reg["model"]
    xmin, xmax = PANEL_B_XLIM
    ymin, ymax = PANEL_B_YLIM
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.figure.canvas.draw()
    x_shift = _axis_x_shift(ax, xmin, ymin)

    x = sub["transmodal_distance"].values
    y = sub["net_SalVentAttn_participation_coefficient_auc"].values
    ax.scatter(
    x + x_shift, y, s=28, c=SN_RED_MID, alpha=0.72,
    edgecolors="white", linewidths=0.4, zorder=3, rasterized=True,
    )

    x1_grid = np.linspace(x.min(), x.max(), 120)
    x_pred = add_constant(
    pd.DataFrame({"transmodal_distance": x1_grid, "unimodal_distance": np.zeros_like(x1_grid)})
    )
    pred = model.get_prediction(x_pred)
    mu = pred.predicted_mean
    ci_lo, ci_hi = pred.conf_int(alpha=0.05).T
    ax.fill_between(
    x1_grid + x_shift, ci_lo, ci_hi,
    color=SN_RED_MID, alpha=0.18, zorder=2, linewidth=0,
    )
    ax.plot(
    x1_grid + x_shift, mu, color=SN_RED_DARK, lw=2.0, alpha=0.92, zorder=4,
    )

    ax.set_xlabel(_title_case_prose("Transmodal distance to SN ($Z$)"))
    ax.set_ylabel(_title_case_prose("SN participation coefficient ($Z$)"))
    ax.set_xticks(PANEL_B_XTICKS + x_shift)
    ax.set_xticklabels([_format_axis_tick(t) for t in PANEL_B_XTICKS])
    ax.set_yticks(PANEL_B_YTICKS)
    ax.set_yticklabels([f"{t:.1f}" for t in PANEL_B_YTICKS])
    ax.set_xlim(xmin, xmax + x_shift)
    ax.figure.canvas.draw()
    _draw_disconnected_axes(ax, xmin, xmax, ymin, ymax, x_shift)
    _outward_tick_marks(ax, xmin, ymin, x_shift)
    _panel_b_forest_inset(ax, model, x_shift)


def panel_c(ax: plt.Axes, corr: dict) -> None:
    sub = corr["sub_z"]
    x = sub.iloc[:, 0].values
    y = sub.iloc[:, 1].values
    xmin, xmax = PANEL_C_XLIM
    ymin, ymax = PANEL_C_YLIM
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.figure.canvas.draw()
    x_shift = _axis_x_shift(ax, xmin, ymin)

    ax.scatter(
    x + x_shift, y, s=28, c=PANEL_C_MID, alpha=0.72,
    edgecolors="white", linewidths=0.4, zorder=3, rasterized=True,
    )

    x_model = add_constant(x)
    ols = OLS(y, x_model).fit()
    x1_grid = np.linspace(x.min(), x.max(), 120)
    pred = ols.get_prediction(add_constant(x1_grid))
    mu = pred.predicted_mean
    ci_lo, ci_hi = pred.conf_int(alpha=0.05).T
    ax.fill_between(
    x1_grid + x_shift, ci_lo, ci_hi,
    color=PANEL_C_MID, alpha=0.18, zorder=2, linewidth=0,
    )
    ax.plot(x1_grid + x_shift, mu, color=PANEL_C_DARK, lw=2.0, alpha=0.92, zorder=4)

    ax.set_xlabel(_title_case_prose("SN gradient dispersion ($Z$)"))
    ax.set_ylabel(_title_case_prose("Susceptibility (High Reserve$\\leftarrow$; $Z$)"))
    ax.set_xticks(PANEL_C_XTICKS + x_shift)
    ax.set_xticklabels([f"{t:.1f}" for t in PANEL_C_XTICKS])
    ax.set_yticks(PANEL_C_YTICKS)
    ax.set_yticklabels([_format_axis_tick(t) for t in PANEL_C_YTICKS])
    ax.set_xlim(xmin, xmax + x_shift)
    ax.figure.canvas.draw()
    _draw_disconnected_axes(ax, xmin, xmax, ymin, ymax, x_shift)
    _outward_tick_marks(ax, xmin, ymin, x_shift)


def _panel_d_jn_curves(mod: dict, w_grid: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    slopes, lo, hi = [], [], []
    for w in w_grid:
        th, _, se = mod["simple_slope"](float(w))
        slopes.append(th)
        lo.append(th - 1.96 * se)
        hi.append(th + 1.96 * se)
    return np.asarray(slopes), np.asarray(lo), np.asarray(hi)


def _panel_d_sn_brain_inset(ax: plt.Axes, bbox: tuple[float, float, float, float], *, dispersed: bool) -> None:
    """Schematic cortical SN layout: compact (high fidelity) vs dispersed (bottleneck)."""
    inset = ax.inset_axes(bbox)
    inset.set_xlim(0, 1)
    inset.set_ylim(0, 1)
    inset.axis("off")
    inset.add_patch(
        mpatches.Ellipse((0.5, 0.5), 0.92, 0.78, fill=False, ec=TEXT, lw=0.65, zorder=1)
    )
    rng = np.random.RandomState(3 if dispersed else 2)
    n = 14
    if dispersed:
        xs = rng.uniform(0.12, 0.88, n)
        ys = rng.uniform(0.14, 0.86, n)
    else:
        xs = 0.50 + rng.normal(0, 0.055, n)
        ys = 0.52 + rng.normal(0, 0.048, n)
    inset.scatter(
        xs, ys, s=10 if dispersed else 14, c=SN_PANEL_A, alpha=0.88 if dispersed else 0.95,
        edgecolors="white", linewidths=0.12, zorder=2,
    )


def _panel_d_region_x_centers(w_bound: float) -> tuple[float, float]:
    x_left = (PANEL_D_WLIM[0] + w_bound) / 2
    x_right = (w_bound + PANEL_D_WLIM[1]) / 2
    return x_left, x_right


def _panel_d_draw_subtitle_lines(
ax: plt.Axes,
trans,
x_data: float,
subtitle: str,
*,
y_first: float,
downward: bool,
subtitle_color: str,
) -> None:
    line1, line2 = (subtitle.split("\n", 1) + [""])[:2]
    gap = PANEL_D_SUBTITLE_INTERNAL_GAP
    if downward:
        ax.text(
        x_data, y_first, line1,
        transform=trans, ha="center", va="top",
        fontsize=FS_SMALL - 0.5, color=subtitle_color, style="italic", zorder=7,
        )
        ax.text(
        x_data, y_first - gap, line2,
        transform=trans, ha="center", va="top",
        fontsize=FS_SMALL - 0.5, color=subtitle_color, style="italic", zorder=7,
        )
    else:
        ax.text(
        x_data, y_first, line1,
        transform=trans, ha="center", va="bottom",
        fontsize=FS_SMALL - 0.5, color=subtitle_color, style="italic", zorder=7,
        )
        ax.text(
        x_data, y_first - gap, line2,
        transform=trans, ha="center", va="bottom",
        fontsize=FS_SMALL - 0.5, color=subtitle_color, style="italic", zorder=7,
        )


def _panel_d_labels_below(
ax: plt.Axes,
x_data: float,
brain_bottom_axes: float,
title: str,
subtitle: str,
*,
title_color: str,
subtitle_color: str,
) -> None:
    trans = blended_transform_factory(ax.transData, ax.transAxes)
    y_title = (
    brain_bottom_axes - PANEL_D_LABEL_GAP + PANEL_D_TEXT_SHIFT_UP
    - PANEL_D_REGION_TEXT_SHIFT_DOWN
    )
    y_sub1 = y_title - PANEL_D_LABEL_TITLE_SUB_GAP
    ax.text(
    x_data, y_title, title,
    transform=trans, ha="center", va="top",
    fontsize=FS_SMALL, fontweight="bold", color=title_color, zorder=7,
    )
    _panel_d_draw_subtitle_lines(
    ax, trans, x_data, subtitle, y_first=y_sub1, downward=True, subtitle_color=subtitle_color,
    )


def _panel_d_labels_above(
ax: plt.Axes,
x_data: float,
brain_top_axes: float,
title: str,
subtitle: str,
*,
title_color: str,
subtitle_color: str,
) -> None:
    trans = blended_transform_factory(ax.transData, ax.transAxes)
    y_sub2 = (
    brain_top_axes + PANEL_D_LABEL_GAP + PANEL_D_TEXT_SHIFT_UP
    - PANEL_D_RIGHT_SUBTITLE_SHIFT_DOWN - PANEL_D_REGION_TEXT_SHIFT_DOWN
    + PANEL_D_RIGHT_LABEL_SHIFT_UP
    )
    y_sub1 = y_sub2 + PANEL_D_SUBTITLE_INTERNAL_GAP
    y_title = y_sub1 + PANEL_D_LABEL_TITLE_SUB_GAP - PANEL_D_RIGHT_TITLE_SHIFT_DOWN
    ax.text(
    x_data, y_title, title,
    transform=trans, ha="center", va="bottom",
    fontsize=FS_SMALL, fontweight="bold", color=title_color, zorder=7,
    )
    _panel_d_draw_subtitle_lines(
    ax, trans, x_data, subtitle, y_first=y_sub1, downward=False, subtitle_color=subtitle_color,
    )


def panel_d(ax: plt.Axes, mod: dict) -> None:
    w_grid = np.linspace(PANEL_D_WLIM[0], PANEL_D_WLIM[1], 400)
    slopes, ci_lo, ci_hi = _panel_d_jn_curves(mod, w_grid)
    w_bound = PANEL_D_W_BOUND
    left = w_grid <= w_bound
    ax.axvspan(w_grid[0], w_bound, color=BLUE_LIGHT, alpha=0.55, zorder=0)
    ax.axvspan(w_bound, w_grid[-1], color=GRAY_LIGHT, alpha=0.65, zorder=0)

    ax.fill_between(w_grid[left], ci_lo[left], ci_hi[left], color=BLUE, alpha=0.28, zorder=1, linewidth=0)
    ax.fill_between(w_grid[~left], ci_lo[~left], ci_hi[~left], color=NET_GRAY, alpha=0.22, zorder=1, linewidth=0)
    ax.axhline(0, color="#4A4A4A", ls=(0, (5.0, 3.5)), lw=1.0, zorder=2)
    ax.plot(w_grid, slopes, color=BLUE, lw=2.4, solid_capstyle="round", zorder=4)
    ax.axvline(w_bound, color=SN_RED_DARK, ls=(0, (4.0, 3.0)), lw=1.2, zorder=3)
    ax.text(
    w_bound + 0.04, PANEL_D_YLIM[1] - 0.04, f"$M_{{Z}} = {w_bound:.3f}$",
    ha="left", va="top", fontsize=FS_SMALL, color=SN_RED_DARK, fontweight="bold",
    )

    ax.set_xlim(*PANEL_D_WLIM)
    ax.set_ylim(*PANEL_D_YLIM)
    ax.set_yticks(PANEL_D_YTICKS)
    ax.set_yticklabels([_format_axis_tick(t) for t in PANEL_D_YTICKS])
    ax.set_xlabel(
    _title_case_prose(
    "Standardized SN Gradient Dispersion\n(Higher Topographic Fidelity$\\leftarrow$; $Z$)"
    )
    )
    ax.set_ylabel(
    _title_case_prose(
    "Simple Slope of Global Modularity ($\\beta$)\n(Predicting RT$_{\\mathrm{Diff}}$)"
    )
    )
    _panel_d_sn_brain_inset(ax, PANEL_D_BRAIN_LEFT, dispersed=False)
    x_left_center, x_right_center = _panel_d_region_x_centers(w_bound)
    _panel_d_labels_below(
    ax, x_left_center, PANEL_D_BRAIN_LEFT[1],
    _title_case_prose("High Fidelity"),
    _title_case_prose("Coupled:\nTopology predicts behavior"),
    title_color=BLUE, subtitle_color="#333333",
    )
    _panel_d_labels_above(
    ax, x_right_center, PANEL_D_BRAIN_RIGHT[1] + PANEL_D_BRAIN_RIGHT[3],
    _title_case_prose("Low Fidelity"),
    _title_case_prose("Decoupled:\nTopology becomes uninformative"),
    title_color=NET_GRAY, subtitle_color="#555555",
    )
    _panel_d_sn_brain_inset(ax, PANEL_D_BRAIN_RIGHT, dispersed=True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def panel_e(
    ax: plt.Axes,
    med: dict,
    *,
    ax_d: plt.Axes | None = None,
    content_dx: float | None = None,
) -> None:
    """Mediation diagram + IMM line - layout aligned with lesion figure panel D."""
    ax.clear()
    ax.set_xlim(*PANEL_E_XLIM)
    ax.set_ylim(*PANEL_E_YLIM)
    ax.axis("off")
    lay = _compute_panel_e_layout(content_dx)
    x0_x = lay["x0_x"]
    x0_m = lay["x0_m"]
    x0_y = lay["x0_y"]
    x0_w = lay["x0_w"]
    y_top = lay["y_top"]
    y0_w = lay["y0_w"]
    bw_x = lay["bw_x"]
    bh = lay["bh"]
    bw_y = lay["bw_y"]
    bh_y = lay["bh_y"]
    bw_w = lay["bw_w"]
    bh_w = lay["bh_w"]
    path_b_mid_x = lay["path_b_mid_x"]
    ox = lay["ox"]
    oy = lay["oy"]
    content_cx = lay["content_cx"]
    content_cy = lay["content_cy"]
    boxes = {
        "x": (x0_x, y_top, bw_x, bh, _title_case_prose("SN participation\ncoefficient (PC)")),
        "m": (x0_m, y_top, bw_x, bh, _title_case_prose("Global\nmodularity")),
        "y": (
            x0_y, y_top, bw_y, bh_y,
            _title_case_prose("Behavioral\ncoordination\n($RT_{\\mathrm{Diff}}$)"),
        ),
        "w": (x0_w, y0_w, bw_w, bh_w, _title_case_prose("SN Gradient\nDispersion")),
    }
    box_linespacing = 1.25
    imm_y = y_top + bh + 0.65
    box_alpha = PANEL_E_LAYOUT["d_box_alpha"]

    def sx(x: float) -> float:
        return x + ox

    def sy(y: float) -> float:
        return y + oy

    for key, (bx, by, w, h, lab) in boxes.items():
        base_fc = BLUE_LIGHT if key == "w" else "white"
        fc = mpl.colors.to_rgba(base_fc, alpha=box_alpha)
        ax.add_patch(
            mpatches.FancyBboxPatch(
                (sx(bx), sy(by)), w, h,
                boxstyle="round,pad=0.02,rounding_size=0.08",
                fc=fc, ec=TEXT, lw=0.9, clip_on=False,
            )
        )
        ax.text(
            sx(bx + w / 2), sy(by + h / 2), lab,
            ha="center", va="center", fontsize=FS, linespacing=box_linespacing,
            clip_on=False,
        )

    def arrow(x1, y1, x2, y2, color=TEXT, lw=1.2):
        ax.annotate(
            "", xy=(sx(x2), sy(y2)), xytext=(sx(x1), sy(y1)),
            arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, shrinkA=0, shrinkB=0),
            clip_on=False,
        )

    y_mid = y_top + bh / 2
    beta_y = y_mid + PANEL_E_LAYOUT["d_beta_y_offset"] - 0.08
    arr_pad = PANEL_E_ARR_PAD
    arrow(x0_x + bw_x + arr_pad, y_mid, x0_m - arr_pad, y_mid, BLUE, 1.6)
    ax.text(
        sx((x0_x + bw_x + x0_m) / 2), sy(beta_y),
        f"$\\beta$ = {med['a1']:.3f}{_sig_stars(med['p_a1'])}",
        ha="center", fontsize=FS_TICK, color=BLUE, clip_on=False,
    )
    arrow(x0_m + bw_x + arr_pad, y_mid, x0_y - arr_pad, y_mid, BLUE, 1.6)
    ax.text(
        sx((x0_m + bw_x + x0_y) / 2), sy(beta_y),
        f"$\\beta$ = {med['b1']:.3f}{_sig_stars(med['p_b1'])}",
        ha="center", fontsize=FS_TICK, color=BLUE, clip_on=False,
    )
    int_arrow_base = y0_w + bh_w + PANEL_E_LAYOUT["d_int_arrow_gap"]
    arrow(path_b_mid_x, int_arrow_base, path_b_mid_x, y_mid, BLUE, 1.6)
    int_label_y = (int_arrow_base + y_mid) / 2
    ax.text(
        sx(path_b_mid_x + PANEL_E_LAYOUT["d_int_label_dx"]), sy(int_label_y),
        f"$\\beta_{{int}}$ = {med['b3']:.3f}{_sig_stars(med['p_b3'])}",
        ha="left", fontsize=FS_TICK, color=BLUE, rotation=90, va="center", clip_on=False,
    )
    ax.text(
        sx(content_cx), sy(imm_y),
        _title_case_prose(
            "Index of moderated mediation = "
            f"{med['imm']:.3f} [95% CI: {med['imm_ci'][0]:.3f}, {med['imm_ci'][1]:.3f}]"
        ),
        ha="center", va="bottom", fontsize=PANEL_E_IMM_FS, color="#333", clip_on=False,
    )
    path_a_mid_x = (x0_x + bw_x + x0_m) / 2
    ci_a_lo, ci_a_hi = med["ci_a1"]
    ax.text(
        sx(path_a_mid_x), sy(content_cy + PANEL_E_LAYOUT["d_path_a_stat_dy"]),
        (
            f"Path $a$: $\\beta$ = {med['a1']:.3f}{_sig_stars(med['p_a1'])} "
            f"[{ci_a_lo:.3f}, {ci_a_hi:.3f}]"
        ),
        ha="center", va="center", fontsize=FS, color="#333", clip_on=False,
    )
    path_lbl_y = sy(y0_w + bh_w + PANEL_E_LAYOUT["d_path_a_lbl_pad"])
    ax.text(
        sx(path_a_mid_x), path_lbl_y, "Hayes Model 14",
        ha="center", va="bottom", fontsize=FS, color="#333", clip_on=False,
    )
    ih_lo, ih_hi = med["ind_hi_ci"]
    il_lo, il_hi = med["ind_lo_ci"]
    note_x = sx(x0_y + bw_y)
    note_line_dy = PANEL_E_LAYOUT["d_note_line_dy"]
    note_y_up = PANEL_E_LAYOUT["d_note_y_up"]
    if ax_d is not None:
        note_y_low = _ax_data_y_from_fig_y(ax, _fig_y_ax_xlabel_bottom(ax.figure, ax_d)) + note_y_up
    else:
        note_y_low = sy(y0_w - 0.72) + note_y_up
    note_y_high = note_y_low + note_line_dy
    ax.text(
        note_x, note_y_low,
        _title_case_prose(
            f"High dispersion (Low Fidelity, $W$ = +1 $\\mathit{{SD}}$): {med['ind_lo']:.3f} "
            f"[95% CI: {il_lo:.3f}, {il_hi:.3f}]"
        ),
        ha="right", va="bottom", fontsize=FS, color="#333", clip_on=False,
    )
    ax.text(
        note_x, note_y_high,
        _title_case_prose(
            f"Low dispersion (High Fidelity, $W$ = -1 $\\mathit{{SD}}$): {med['ind_hi']:.3f} "
            f"[95% CI: {ih_lo:.3f}, {ih_hi:.3f}]"
        ),
        ha="right", va="bottom", fontsize=FS, color="#333", clip_on=False,
    )
